In [1]:
import pyodbc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
!pip install scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler


In [2]:
conn=pyodbc.connect("DRIVER={ODBC driver 17 for SQl Server};"
                    "SERVER=.\SQLEXPRESS;"
                    "DATABASE=Banking_Analysis;"
                    "Trusted_connection=yes;")
                    

In [3]:
client_df=pd.read_sql("Select * from client_360",conn)

In [4]:
trans_df=pd.read_sql("select * from master_transaction",conn)

In [5]:
account_df=pd.read_sql("select * from account",conn)

In [6]:
disp_df=pd.read_sql("select * from disp",conn)

In [7]:
client_df.head()

,client_id,age,gender,district,have_account_flag,total_account,owner_account_flag,user_account_flag,saving_account_flag,salary_account_flag,NRI_account_flag,avg_balance,card_type,active_loan_flag,account_opening_date,total_transaction_per_month,monthly_fee_flag,weekly_fee_flag,transaction_fee_flag,last_trans_date
0,1,55,female,Pisek,True,1,1,0,0,0,1,13413.9,No card,False,2018-03-24,5,1,0,0,2021-12-13
1,2,81,male,Hl.m. Praha,True,1,1,0,0,1,0,42471.3,No card,False,2016-02-26,6,1,0,0,2021-12-17
2,3,85,female,Hl.m. Praha,True,1,0,1,0,1,0,42471.3,No card,False,2016-02-26,6,1,0,0,2021-12-17
3,4,69,male,Kolin,True,1,1,0,0,0,1,53446.5,No card,False,2020-07-07,6,1,0,0,2021-12-11
4,5,65,female,Kolin,True,1,0,1,0,0,1,53446.5,No card,False,2020-07-07,6,1,0,0,2021-12-11


In [8]:
trans_df.head()

,trans_id,account_id,Date,Type,operation,amount,balance,Purpose,bank,account_partern_id,year,month,month_no
0,732436,2503,2016-12-31,Withdrawal,Withdrawal in cash,15,27875.500000,Payment on Statement,NaN,Not applicable,2016,December,12
1,730715,2497,2016-12-31,Withdrawal,Withdrawal in cash,15,18709.900391,Payment on Statement,NaN,Not applicable,2016,December,12
2,733937,2508,2016-12-31,Withdrawal,Withdrawal in cash,15,21579.099609,Payment on Statement,NaN,Not applicable,2016,December,12
3,802764,2736,2016-12-31,Withdrawal,Withdrawal in cash,15,32351.400391,Payment on Statement,NaN,Not applicable,2016,December,12
4,801171,2732,2016-12-31,Withdrawal,Withdrawal in cash,15,65012.500000,Payment on Statement,NaN,Not applicable,2016,December,12


In [9]:
account_df

,account_id,district_id,frequency,date,Account_type,final_date
0,1573,63,POPLATEK MESICNE,1997-12-29,Savings account,2020-12-29
1,3276,1,POPLATEK MESICNE,1997-12-29,Salary account,2020-12-29
2,124,55,POPLATEK MESICNE,1997-12-28,Salary account,2020-12-28
3,3958,59,POPLATEK MESICNE,1997-12-28,NRI account,2020-12-28
4,777,30,POPLATEK MESICNE,1997-12-28,Savings account,2020-12-28
...,...,...,...,...,...,...
4495,1972,77,POPLATEK MESICNE,1993-01-02,NRI account,2016-01-02
4496,576,55,POPLATEK MESICNE,1993-01-01,Savings account,2016-01-01
4497,3818,74,POPLATEK MESICNE,1993-01-01,NRI account,2016-01-01
4498,704,55,POPLATEK MESICNE,1993-01-01,Salary account,2016-01-01


In [10]:
df=pd.merge(left=trans_df,right=account_df,left_on='account_id',right_on='account_id',how='inner')

In [11]:
df=pd.merge(left=df, right=disp_df, left_on='account_id' , right_on='account_id', how='left')

In [12]:
df=pd.merge(left=df, right=client_df, left_on='client_id', right_on='client_id', how='left')

In [13]:
df

,trans_id,account_id,Date,Type,operation,amount,balance,Purpose,bank,account_partern_id,...,NRI_account_flag,avg_balance,card_type,active_loan_flag,account_opening_date,total_transaction_per_month,monthly_fee_flag,weekly_fee_flag,transaction_fee_flag,last_trans_date
0,732436,2503,2016-12-31,Withdrawal,Withdrawal in cash,15,27875.500000,Payment on Statement,NaN,Not applicable,...,1,66916.8,No card,False,2016-03-05,5,1,0,0,2021-12-14
1,732436,2503,2016-12-31,Withdrawal,Withdrawal in cash,15,27875.500000,Payment on Statement,NaN,Not applicable,...,1,66916.8,No card,False,2016-03-05,5,1,0,0,2021-12-14
2,730715,2497,2016-12-31,Withdrawal,Withdrawal in cash,15,18709.900391,Payment on Statement,NaN,Not applicable,...,0,38973.9,No card,False,2016-03-16,5,1,0,0,2021-12-17
3,733937,2508,2016-12-31,Withdrawal,Withdrawal in cash,15,21579.099609,Payment on Statement,NaN,Not applicable,...,0,18874.0,No card,False,2016-03-14,4,1,0,0,2021-12-10
4,802764,2736,2016-12-31,Withdrawal,Withdrawal in cash,15,32351.400391,Payment on Statement,NaN,Not applicable,...,0,29212.5,No card,False,2016-07-02,5,1,0,0,2021-12-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253397,490298,1673,2021-01-01,Withdrawal,Withdrawal in cash,1120,14583.599609,NaN,NaN,Not applicable,...,0,13077.0,No card,False,2016-12-08,4,1,0,0,2021-12-12
1253398,488812,1667,2021-01-01,Withdrawal,Withdrawal in cash,2640,26219.000000,NaN,NaN,Not applicable,...,0,17709.8,No card,False,2019-08-18,4,1,0,0,2021-12-12
1253399,488812,1667,2021-01-01,Withdrawal,Withdrawal in cash,2640,26219.000000,NaN,NaN,Not applicable,...,0,17709.8,No card,False,2019-08-18,4,1,0,0,2021-12-12
1253400,518574,1771,2021-01-01,Withdrawal,Withdrawal in cash,800,24267.599609,NaN,NaN,Not applicable,...,0,23272.1,No card,False,2016-03-28,5,1,0,0,2021-12-11


In [14]:
df.columns

Index(['trans_id', 'account_id', 'Date', 'Type', 'operation', 'amount',
       'balance', 'Purpose', 'bank', 'account_partern_id', 'year', 'month',
       'month_no', 'district_id', 'frequency', 'date', 'Account_type',
       'final_date', 'disp_id', 'client_id', 'type', 'age', 'gender',
       'district', 'have_account_flag', 'total_account', 'owner_account_flag',
       'user_account_flag', 'saving_account_flag', 'salary_account_flag',
       'NRI_account_flag', 'avg_balance', 'card_type', 'active_loan_flag',
       'account_opening_date', 'total_transaction_per_month',
       'monthly_fee_flag', 'weekly_fee_flag', 'transaction_fee_flag',
       'last_trans_date'],
      dtype='str')

In [15]:
df = df.drop(columns=['trans_id','account_id','client_id','disp_id','Type','type','operation','Purpose','bank','account_partern_id',
                      'Date','date','final_date','year','month','month_no','balance','district_id','frequency','Account_type'])

In [16]:
df['estimated_income'] = df['avg_balance'] * 2   # approx logic
df['emi_capacity'] = df['estimated_income'] * 0.3

In [17]:
df['loan_eligibility'] = ((df['avg_balance'] > 5000) &(df['total_transaction_per_month'] > 5) &(df['salary_account_flag'] == 1) &
    (df['active_loan_flag'] == 0) &(df['emi_capacity'] > 2000)).astype(int)

In [18]:
X = df[['avg_balance','total_transaction_per_month','total_account','salary_account_flag','saving_account_flag','age',
    'gender','card_type','monthly_fee_flag','transaction_fee_flag','emi_capacity']]

In [19]:
X = pd.get_dummies(X, drop_first=True)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, df['loan_eligibility'], test_size=0.2, random_state=42)

In [21]:


scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [22]:
model = LogisticRegression()
model=model.fit(X_train, y_train)

In [23]:
train_pred=  model.predict(X_train)
test_pred=model.predict(X_test)

In [24]:
print(metrics.classification_report(y_train,train_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98    891467
           1       0.82      0.79      0.81    111254

    accuracy                           0.96   1002721
   macro avg       0.90      0.89      0.89   1002721
weighted avg       0.96      0.96      0.96   1002721



In [25]:
print(metrics.classification_report(y_test,test_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98    222703
           1       0.82      0.79      0.80     27978

    accuracy                           0.96    250681
   macro avg       0.90      0.88      0.89    250681
weighted avg       0.96      0.96      0.96    250681



In [26]:
import joblib

In [28]:
joblib.dump(model,'banking_model.pkl')
joblib.dump(scaler, 'banking_scaler.pkl')

['banking_scaler.pkl']

'streamlit' is not recognized as an internal or external command,
operable program or batch file.
